In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

import mlflow
import dagshub
import mlflow.sklearn



In [ ]:
# MLflow
dagshub.init(
    repo_owner='IzaKakhniashvili',
    repo_name='ML-assignment1-HousePrices',
    mlflow=True
)


In [ ]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [ ]:
train.shape, test.shape

In [ ]:
train.info()

In [ ]:
train["SalePrice"].describe()

In [ ]:
train.isnull().sum().sort_values(ascending=False).head(20)

## Baseline

In [ ]:
y = train["SalePrice"]
X = train.drop("SalePrice", axis=1)


In [ ]:
# mode/median ით დავჰენდლეთ ყველაზე მარტივად
for col in X.columns:
    if X[col].dtype == "object":
        X[col] = X[col].fillna(X[col].mode()[0])
    else:
        X[col] = X[col].fillna(X[col].median())

In [ ]:
X = pd.get_dummies(X)
X

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
rmse

In [ ]:
r2 = r2_score(y_val, preds)
r2

In [ ]:
mae = mean_absolute_error(y_val, preds)
mae

In [ ]:
import mlflow

with mlflow.start_run():
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("stage", "baseline")

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

## Ex1_Categorial Expansion

### Data Cleaning

In [ ]:
X = train.drop("SalePrice", axis=1)
y = train["SalePrice"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_tr = X_train.copy()
X_v = X_val.copy()

In [ ]:
num_cols = X_tr.select_dtypes(include=[np.number]).columns
X_tr[num_cols] = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_v[num_cols] = X_v[num_cols].fillna(X_tr[num_cols].median())

In [ ]:
cat_cols = X_tr.select_dtypes(include=['object']).columns
X_tr[cat_cols] = X_tr[cat_cols].fillna("None")
X_v[cat_cols] = X_v[cat_cols].fillna("None")

### Feature Engineering

In [ ]:
for df in [X_tr, X_v]:
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']

df

In [ ]:
X_tr['is_train'] = 1
X_v['is_train'] = 0
combined = pd.concat([X_tr, X_v])

combined_encoded = pd.get_dummies(combined, columns=cat_cols)

X_tr_enc = combined_encoded[combined_encoded['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined_encoded[combined_encoded['is_train'] == 0].drop('is_train', axis=1)

### Feature Selection

In [ ]:
temp_df = pd.concat([X_tr_enc, y_train], axis=1)
corr_matrix = temp_df.corr()

threshold = 0.4
relevant_cols = corr_matrix['SalePrice'][abs(corr_matrix['SalePrice']) > threshold].index.tolist()
relevant_cols
